In [1]:
SCIPY_ARRAY_API=1
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import pandas as pd

In [2]:
import os
import pickle as pkl

from pydeseq2.dds import DeseqDataSet
from pydeseq2.default_inference import DefaultInference
from pydeseq2.ds import DeseqStats


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.4 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/Users/beatricecitterio/opt/anaconda3/envs/ml/lib/python3.12/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/Users/beatricecitterio/opt/anaconda3/envs/ml/lib/python3.12/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/Users/beatricecitterio/opt/anaconda3/envs/ml/lib/python3.12/site-packages/ipykernel/kernelapp

In [3]:
mutation = pd.read_csv('OmicsSomaticMutations.csv')
expression = pd.read_csv('OmicsExpressionProteinCodingGenesTPMLogp1.csv')
info = pd.read_csv('Model.csv')
expression = expression.rename(columns={'Unnamed: 0': 'ModelID'})
mutation = mutation[mutation.HugoSymbol == 'TP53'] # filter for TP53 mutations
expression['Mutation'] = expression['ModelID'].isin(mutation['ModelID']).astype(int) 
# here we only consider whether there is mutation or not

/var/folders/bm/yxd_5px52t5d6m_z9t3mp9400000gn/T/ipykernel_4649/1488345977.py:1: DtypeWarning: Columns (22,50,56,57,58,59,61) have mixed types. Specify dtype option on import or set low_memory=False.
  mutation = pd.read_csv('OmicsSomaticMutations.csv')


In [4]:
expression.drop(columns=['ModelID'], inplace=True)

In [5]:
expression['Mutation'] = expression['Mutation'].map({0: "WT", 1: "MUT"}).astype("category")

### **Filter Data**

In [6]:
samples_to_keep = ~expression.Mutation.isna()
expression = expression.loc[samples_to_keep]

In [ ]:
# genes_to_keep = expression.columns[expression.sum(axis=0) >= 10]
# expression = expression[genes_to_keep]

### **Single Factor Analysis**
In this first analysis, we ignore use the mutation column as our design factor. That is, we compare gene expressions of samples that have mutation to those that don't. We start by creating a DeseqDataSet object from the data. A DeseqDataSet fits dispersion and log-fold change (LFC) parameters from the data, and stores them.

In [7]:
inference = DefaultInference(n_cpus=8)
dds = DeseqDataSet(
    counts=expression.drop(columns=['Mutation']).astype(int),
    metadata=expression[['Mutation']],
    design="~Mutation",
    refit_cooks=True,
    inference=inference,
    # n_cpus=8, # n_cpus can be specified here or in the inference object
)

/Users/beatricecitterio/opt/anaconda3/envs/ml/lib/python3.12/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


Once a DeseqDataSet was initialized, we may run the deseq2() method to fit dispersions and LFCs.

In [8]:
dds.deseq2()


Fitting size factors...


Using None as control genes, passed at DeseqDataSet initialization


... done in 0.77 seconds.

python(8418) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(8419) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(8420) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(8421) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(8422) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(8423) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(8424) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(8425) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(8426) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(8427) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Fit

Parameters are stored according to the AnnData data structure, with key-based data fields. In particular,
- X stores the count data,
- obs stores design factors,
- obsm stores sample-level data, such as "design_matrix" and "size_factors",
- varm stores gene-level data, such as "dispersions" and "LFC".

Here is how we would access dispersions and LFCs (in natural log scale):


In [9]:
print(dds.varm["dispersions"])

[3.30446463e-03 1.50454265e+02 3.85611086e-04 ...            nan
 1.67300000e+03 6.58893672e+02]


In [10]:
print(dds.varm["LFC"])


                    Intercept  Mutation[T.WT]
TSPAN6 (7105)        1.110076       -0.106856
TNMD (64102)        -1.645630        0.084062
DPM1 (8813)          1.787978       -0.014901
SCYL3 (57147)        0.601944        0.009544
C1orf112 (55732)     1.147103       -0.042404
...                       ...             ...
ELOA3B (728929)           NaN             NaN
NPBWR1 (2831)       -1.355570       -0.194354
ELOA3D (100506888)        NaN             NaN
ELOA3 (162699)      -1.706480        0.010996
CDR1 (1038)         -1.696339        0.021234

[19193 rows x 2 columns]


Now that dispersions and LFCs were fitted, we may proceed with statistical tests to compute p-values and adjusted p-values for differential expresion. This is the role of the DeseqStats class. It has two mandatory arguments:
- dds, which should be a fitted DeseqDataSet object,
- contrast, which is a list of three strings of the form ["variable", "tested_level", "control_level"], or directly a contrast vector.

In [11]:
ds = DeseqStats(dds, contrast=["Mutation", "MUT", "WT"], inference=inference)

PyDESeq2 computes p-values using Wald tests. This can be done using the summary() method, which runs the whole statistical analysis, cooks filtering and multiple testing adjustement included.

In [12]:
ds.summary()

Running Wald tests...


Log2 fold change & Wald test p-value: Mutation MUT vs WT
                    baseMean  log2FoldChange     lfcSE      stat    pvalue  \
TSPAN6 (7105)       2.909326        0.154160  0.042503  3.627019  0.000287   
TNMD (64102)        0.046113       -0.121276  0.893680 -0.135704  0.892055   
DPM1 (8813)         5.952193        0.021498  0.029345  0.732579  0.463815   
SCYL3 (57147)       1.822844       -0.013769  0.052717 -0.261184  0.793951   
C1orf112 (55732)    3.072844        0.061176  0.040768  1.500585  0.133463   
...                      ...             ...       ...       ...       ...   
ELOA3B (728929)     0.000000             NaN       NaN       NaN       NaN   
NPBWR1 (2831)       0.131317        0.280393  0.334623  0.837939  0.402065   
ELOA3D (100506888)  0.000000             NaN       NaN       NaN       NaN   
ELOA3 (162699)      0.000569       -0.015863  2.937361 -0.005401  0.995691   
CDR1 (1038)         0.007695       -0.030634  1.847870 -0.016578  0.986773   

      

... done in 1.38 seconds.



The results are then stored in the results_df attribute (ds.results_df).

The gene TSPAN6 is significantly upregulated in TP53 MUT samples.
log2FC = 0.15 ⇒ ~11% increase (2^0.15 ≈ 1.11).
padj < 0.05 ⇒ considered statistically significant.


In [13]:
degs = ds.results_df[ds.results_df['padj'] < 0.05]
degs = degs.sort_values(by = 'log2FoldChange', ascending = False)

degs.to_csv('DEGs_filtered_sorted.csv', index=True)


In [ ]:
top_genes = degs.sort_values(by='log2FoldChange', ascending=False)


In [ ]:
top_genes.sort_values(by='padj', ascending=True, inplace=True)

In [ ]:
results = ds.results_df

| Column Name      | Meaning                                                               |
| ---------------- | --------------------------------------------------------------------- |
| `baseMean`       | Mean of normalized counts for the gene across all samples.            |
| `log2FoldChange` | Estimated **log₂ fold change** (MUT vs WT). Positive = higher in MUT. |
| `lfcSE`          | Standard error of the log₂ fold change estimate.                      |
| `stat`           | Wald test statistic for differential expression.                      |
| `pvalue`         | Raw p-value from the Wald test.                                       |
| `padj`           | Adjusted p-value (FDR, Benjamini-Hochberg corrected).                 |


In [ ]:
results.sort_values(by ='padj', ascending=True, inplace=True)

In [ ]:
results

,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj
EDA2R (60401),0.806853,-2.592502,0.110485,-23.464744,9.348125e-122,1.406519e-117
CDKN1A (1026),4.847723,-0.563770,0.032016,-17.608770,2.109785e-69,1.587192e-65
ACTA2 (59),2.613672,-0.966712,0.063594,-15.201386,3.462187e-52,1.736402e-48
ZMAT3 (64393),2.285987,-0.690544,0.046477,-14.857632,6.208078e-50,2.335168e-46
MDM2 (4193),4.481441,-0.461569,0.033311,-13.856543,1.161348e-43,3.494727e-40
...,...,...,...,...,...,...
ELOA3B (728929),0.000000,NaN,NaN,NaN,NaN,NaN
NPBWR1 (2831),0.131317,0.280393,0.334623,0.837939,4.020652e-01,NaN
ELOA3D (100506888),0.000000,NaN,NaN,NaN,NaN,NaN
ELOA3 (162699),0.000569,-0.015863,2.937361,-0.005401,9.956910e-01,NaN


In [ ]:
pos_corr = degs[:10]

neg_corr = degs[-10:]

In [ ]:
pos_corr

,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj
KLK5 (25818),0.796674,1.833900,0.247747,7.402318,1.338280e-13,5.412840e-12
VGLL1 (51442),0.844664,1.630179,0.229722,7.096308,1.281339e-12,4.401605e-11
KLK8 (11202),1.014336,1.627846,0.196860,8.269046,1.350348e-16,9.538653e-15
KLK10 (5655),1.057977,1.583079,0.185490,8.534577,1.406696e-17,1.119849e-15
KLK6 (5653),1.271117,1.579358,0.167620,9.422246,4.415392e-21,6.710503e-19
KRT4 (3851),0.722298,1.561718,0.232209,6.725493,1.749987e-11,5.092903e-10
ADGRF1 (266977),1.125959,1.532975,0.174966,8.761552,1.925805e-18,1.777648e-16
ALPP (250),0.792395,1.499744,0.201791,7.432157,1.068412e-13,4.392164e-12
GRHL2 (79977),1.215151,1.498595,0.152515,9.825858,8.712924e-23,1.872781e-20
KRT13 (3860),0.957701,1.477998,0.223002,6.627720,3.409122e-11,9.326118e-10


In [ ]:
neg_corr

,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj
SGCD (6444),0.513989,-1.470225,0.201347,-7.301949,2.836284e-13,1.095445e-11
S100B (6285),0.760793,-1.507430,0.257172,-5.861564,4.585288e-09,8.444338e-08
HSPB2 (3316),0.539872,-1.551862,0.237459,-6.535289,6.348662e-11,1.655493e-09
SOX10 (6663),0.517571,-1.571731,0.431699,-3.640803,2.717893e-04,1.747582e-03
TYR (7299),0.414011,-1.623530,0.525584,-3.089004,2.008287e-03,9.998904e-03
PLP1 (5354),0.614418,-1.625163,0.316643,-5.132477,2.859532e-07,3.588367e-06
MLANA (2315),0.428437,-1.645855,0.450706,-3.651728,2.604821e-04,1.682066e-03
SPATA18 (132671),0.436036,-1.796310,0.156887,-11.449673,2.360335e-30,2.219600e-27
COX7A1 (1346),0.512022,-1.809034,0.358969,-5.039528,4.666824e-07,5.639923e-06
EDA2R (60401),0.806853,-2.592502,0.110485,-23.464744,9.348125e-122,1.406519e-117


## Build Classifier only on statistically relevant genes


In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectFromModel, RFE
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc, roc_auc_score
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA

In [ ]:
mutation = pd.read_csv('OmicsSomaticMutations.csv')
expression = pd.read_csv('OmicsExpressionProteinCodingGenesTPMLogp1.csv')
info = pd.read_csv('Model.csv')
expression = expression.rename(columns={'Unnamed: 0': 'ModelID'})
mutation = mutation[mutation.HugoSymbol == 'TP53'] # filter for TP53 mutations
expression['Mutation'] = expression['ModelID'].isin(mutation['ModelID']).astype(int) 

/var/folders/bm/yxd_5px52t5d6m_z9t3mp9400000gn/T/ipykernel_5115/3728035584.py:1: DtypeWarning: Columns (22,50,56,57,58,59,61) have mixed types. Specify dtype option on import or set low_memory=False.
  mutation = pd.read_csv('OmicsSomaticMutations.csv')


In [ ]:
X = expression[expression.columns.intersection(degs.index)]
y = expression['Mutation'].astype('category').cat.codes

In [ ]:
# Keep column names for later feature importance analysis
feature_names = X.columns.tolist()

# 4. Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 5. Display class distribution
print(f"Training set: Class 0 (WT): {sum(y_train==0)}, Class 1 (Mutated): {sum(y_train==1)}")
print(f"Testing set: Class 0 (WT): {sum(y_test==0)}, Class 1 (Mutated): {sum(y_test==1)}")

Training set: Class 0 (WT): 549, Class 1 (Mutated): 789
Testing set: Class 0 (WT): 138, Class 1 (Mutated): 197


In [ ]:
def evaluate_model(trained_model, X_test, y_test):
    y_pred = trained_model.predict(X_test)
    print("Confusion Matrix:")  
    print(confusion_matrix(y_test, y_pred))
    print("Classification Report:")
    print(classification_report(y_test, y_pred))
    print("ROC AUC Score:", roc_auc_score(y_test, trained_model.predict_proba(X_test)[:, 1]))

    return None

In [ ]:

from sklearn.linear_model import LogisticRegression

clf = LogisticRegression(random_state=0)
clf.fit(X_train, y_train)

evaluate_model(clf, X_test, y_test)

Confusion Matrix:
[[113  25]
 [ 19 178]]
Classification Report:
              precision    recall  f1-score   support

           0       0.86      0.82      0.84       138
           1       0.88      0.90      0.89       197

    accuracy                           0.87       335
   macro avg       0.87      0.86      0.86       335
weighted avg       0.87      0.87      0.87       335

ROC AUC Score: 0.9196645332156257


/Users/beatricecitterio/opt/anaconda3/envs/ml/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train) 

evaluate_model(rf, X_test, y_test)

Confusion Matrix:
[[110  28]
 [ 23 174]]
Classification Report:
              precision    recall  f1-score   support

           0       0.83      0.80      0.81       138
           1       0.86      0.88      0.87       197

    accuracy                           0.85       335
   macro avg       0.84      0.84      0.84       335
weighted avg       0.85      0.85      0.85       335

ROC AUC Score: 0.9102111380857795


In [ ]:
models = {}

# Logistic Regression
from sklearn.linear_model import LogisticRegression
models['Logistic Regression'] = LogisticRegression()

# Support Vector Machines
from sklearn.svm import LinearSVC
models['Support Vector Machines'] = LinearSVC()

# Decision Trees
from sklearn.tree import DecisionTreeClassifier
models['Decision Trees'] = DecisionTreeClassifier()

# Random Forest
from sklearn.ensemble import RandomForestClassifier
models['Random Forest'] = RandomForestClassifier()

# Naive Bayes
from sklearn.naive_bayes import GaussianNB
models['Naive Bayes'] = GaussianNB()

# K-Nearest Neighbors
from sklearn.neighbors import KNeighborsClassifier
models['K-Nearest Neighbor'] = KNeighborsClassifier()

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score

accuracy, precision, recall = {}, {}, {}

for key in models.keys():
    
    # Fit the classifier
    models[key].fit(X_train, y_train)
    
    # Make predictions
    predictions = models[key].predict(X_test)
    
    # Calculate metrics
    accuracy[key] = accuracy_score(predictions, y_test)
    precision[key] = precision_score(predictions, y_test)
    recall[key] = recall_score(predictions, y_test)

/Users/beatricecitterio/opt/anaconda3/envs/ml/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/Users/beatricecitterio/opt/anaconda3/envs/ml/lib/python3.12/site-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/Users/beatricecitterio/opt/anaconda3/envs/ml/lib/python3.12/site-packages/threadpoolctl.py:1226: RuntimeWarning: 
Found Intel OpenMP ('libiomp') and LLVM OpenMP ('libomp') loaded at
the same time. Both libraries are known to be incompatible and this

In [ ]:
import pandas as pd

df_model = pd.DataFrame(index=models.keys(), columns=['Accuracy', 'Precision', 'Recall'])
df_model['Accuracy'] = accuracy.values()
df_model['Precision'] = precision.values()
df_model['Recall'] = recall.values()

df_model

,Accuracy,Precision,Recall
Logistic Regression,0.868657,0.903553,0.876847
Support Vector Machines,0.853731,0.883249,0.870000
Decision Trees,0.788060,0.776650,0.850000
Random Forest,0.835821,0.878173,0.848039
Naive Bayes,0.695522,0.619289,0.818792
K-Nearest Neighbor,0.746269,0.781726,0.785714


Now we compare on the whole dataset

In [ ]:
X = expression.drop(columns=['ModelID', 'Mutation'])
y = expression['Mutation'].astype('category').cat.codes

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
models = {}

# Logistic Regression
from sklearn.linear_model import LogisticRegression
models['Logistic Regression'] = LogisticRegression()

# Support Vector Machines
from sklearn.svm import LinearSVC
models['Support Vector Machines'] = LinearSVC()

# Decision Trees
from sklearn.tree import DecisionTreeClassifier
models['Decision Trees'] = DecisionTreeClassifier()

# Random Forest
from sklearn.ensemble import RandomForestClassifier
models['Random Forest'] = RandomForestClassifier()

# Naive Bayes
from sklearn.naive_bayes import GaussianNB
models['Naive Bayes'] = GaussianNB()

# K-Nearest Neighbors
from sklearn.neighbors import KNeighborsClassifier
models['K-Nearest Neighbor'] = KNeighborsClassifier()
from sklearn.metrics import accuracy_score, precision_score, recall_score

accuracy, precision, recall = {}, {}, {}

for key in models.keys():
    
    # Fit the classifier
    models[key].fit(X_train, y_train)
    
    # Make predictions
    predictions = models[key].predict(X_test)
    
    # Calculate metrics
    accuracy[key] = accuracy_score(predictions, y_test)
    precision[key] = precision_score(predictions, y_test)
    recall[key] = recall_score(predictions, y_test)
import pandas as pd

df_model = pd.DataFrame(index=models.keys(), columns=['Accuracy', 'Precision', 'Recall'])
df_model['Accuracy'] = accuracy.values()
df_model['Precision'] = precision.values()
df_model['Recall'] = recall.values()

df_model

/Users/beatricecitterio/opt/anaconda3/envs/ml/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/Users/beatricecitterio/opt/anaconda3/envs/ml/lib/python3.12/site-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


,Accuracy,Precision,Recall
Logistic Regression,0.877612,0.893401,0.897959
Support Vector Machines,0.862687,0.878173,0.887179
Decision Trees,0.776119,0.802030,0.814433
Random Forest,0.832836,0.873096,0.847291
Naive Bayes,0.704478,0.700508,0.775281
K-Nearest Neighbor,0.728358,0.725888,0.794444


## **Target Genes**

In [ ]:
target_genes = 'CDKN1A	ABCA12	NTPCR	PGPEP1	RNF19B	LCE1E	EPN3	SCN4B	ARVCF	FCHO2	PANK2\tTMEM8B\tRRM2B	ANKRA2	ORAI3	PLCL2	SAC3D1	LIMK2	FBXO32	SCRIB	BHLHE40	FCHSD2	PAQR7	TP53   \tMDM2	CCNG1	PRKAB1	PMAIP1	SYTL1	LRP1	FHL2	SEMA3B	BMP7	FLRT2	PCBP4	TP53I11\tSUSD6	CYFIP2	PTP4A1	PRDM1	TNFRSF10A	MCC	HES2	SLC25A45	BORCS7	GBE1	PERP	TRAK1\tDF15	DRAM1	SESN2	RAP2B	TNFRSF10D	NUPR1	KCNN4	SLC44A5\tBTBD10	GPC1	PLLP	TRIP6\tBTG2	FBXO22	SLC30A1	RRAD	TSPAN11	PARD6G	KLHDC7A	SLC4A11	BTG3	HES1	POU3F1	TSGA10 DDB2	ISCU	SPATA18	ZNF219	VWCE	PHPT1	LMNA	SLC9A1	C17orf89	HRAS	PPFIBP1	UNC5B\tGADD45A	PHLDA3	TGFA	ZNF337	DDIT4	PIDD1	MLF2	STAT3	CAPN2	HSD17B3	PPM1J	UQCC1\tPLK3 	SERPINB5	TLR3	ACTA2	RAD51C	PML	MR1	STK17A	CASP6	ICOSLG	PPP4R3A	VDR\tTIGAR	SERTAD1	TM7SF3	EDN2	SERPINE1	PTPRE	MYO6	STX6	CATSPERG	IGFBP7	PTAFR	YPEL3\tRPS27L	TRAF4	TMEM68	ALOX5	TNFAIP8	PVRL4	NEFL	TP73	CAV1	IL1B	RALGDS	ZNF195\tTNFRSF10B	TRIM22	WDR63	ARHGEF3	TSKU	RETSAT	NKAIN4	TRIM32	CCNK	ISYNA1	RBM38	ZNF385A TRIAP1	CES2	ZNF561	CERS5	PCNA	REV3L	PCLO	TRIM38	CFLAR	JAG1	RGL1	ZNF488\tZMAT3	CMBL	ZNF79	DDR1	ACYP2	RNASE7	PDE4C	TRIM5	CGB7	KRT8	RGS20\tBAX	FBXW7	ASCC3	DHRS3	APAF1	SFN	PGAP1	TYMSOS	CHST14	KSR1	RHOC	PGF	HSPA4L	ACER2	DUSP14	APOBEC3H	TNFRSF10C	PLCXD2	AKAP9	COBLL1	LACC1	RPS19	POLH	KITLG	ANXA4	E2F7	BCL2L1	TRIML2	PLEKHF1	CCDC51	CPEB2	LPXN	SARS	PPM1D	SLC12A4	APOBEC3C	EPS8L2	BCL6	VCAN	PLTP	CDH8	CPSF4	LRPAP1	SCIN	SULF2	ATF3	ASTN2	FAM210B	BLCAP	ADCK3	PLXNB1	DUSP11	DNAJB2	MFAP3L	SCN3B	 XPC	BBC3	CD82	GLS2	C17orf82	AK3	PLXNB2	GCC2	DOCK8	MKNK2	SDC4	AEN	CCDC90B	CDIP1	GPX1	COL7A1	ALDH1A3	PRKAB2	METTL8	DUSP5	MON2	SDPR	BLOC1S2	DYRK3	CPE	GRHL3	CPEB4	BBS2	PRKX	PPP1R3C	DUSP7	MRPL49	SMAD3	FAS	EDA2R	CSF1	HHAT	CSNK1G1	BTG1	PRODH	STEAP3	EBI3	MYBPHL	SNX2	GPR87	EPHA2	DCP1B	IGDCC4	DGKA	CEL	PTPRU	ABHD4	EFNB1	MYLK	SOCS4	NINJ1	FAM13C	ENC1	IKBIP	FAM49A	CLCA2	RGMA	ABTB2	EI24	MYOF	TAB3	PLK2	FAM198B	FOSL1	LAPTM5	FAM84B	CLDN1	RGS16	ADGRG1	EML2	NFKBIA	TCAIM	PSTPIP2	FAM212B	FUCA1	MAST4	GNAI1	CLP1	RND3	AIFM2	ENPP2	NHLH2	TEP1	SESN1	FDXR	IER5	MICALL1	INPP1	CROT	RNF144B	AMOTL1	ETV7	NLRP1	TET2	TP53I3	LIF	PADI4	NOTCH1	ITGA3	CYP4F3	S100A2	AMZ2	FAM196A	NYNRIN	TEX9	TP53INP1	NADSYN1	PANK1	RABGGTA	KRT15	DAPK1	SCN2A	ARC	FAM98C	P3H2	TMEM63B'


In [ ]:
target_genes = target_genes.split('\t')

In [ ]:
target_genes

['CDKN1A',
 'ABCA12',
 'NTPCR',
 'PGPEP1',
 'RNF19B',
 'LCE1E',
 'EPN3',
 'SCN4B',
 'ARVCF',
 'FCHO2',
 'PANK2',
 'TMEM8B',
 'RRM2B',
 'ANKRA2',
 'ORAI3',
 'PLCL2',
 'SAC3D1',
 'LIMK2',
 'FBXO32',
 'SCRIB',
 'BHLHE40',
 'FCHSD2',
 'PAQR7',
 'TP53   ',
 'MDM2',
 'CCNG1',
 'PRKAB1',
 'PMAIP1',
 'SYTL1',
 'LRP1',
 'FHL2',
 'SEMA3B',
 'BMP7',
 'FLRT2',
 'PCBP4',
 'TP53I11',
 'SUSD6',
 'CYFIP2',
 'PTP4A1',
 'PRDM1',
 'TNFRSF10A',
 'MCC',
 'HES2',
 'SLC25A45',
 'BORCS7',
 'GBE1',
 'PERP',
 'TRAK1',
 'DF15',
 'DRAM1',
 'SESN2',
 'RAP2B',
 'TNFRSF10D',
 'NUPR1',
 'KCNN4',
 'SLC44A5',
 'BTBD10',
 'GPC1',
 'PLLP',
 'TRIP6',
 'BTG2',
 'FBXO22',
 'SLC30A1',
 'RRAD',
 'TSPAN11',
 'PARD6G',
 'KLHDC7A',
 'SLC4A11',
 'BTG3',
 'HES1',
 'POU3F1',
 'TSGA10 DDB2',
 'ISCU',
 'SPATA18',
 'ZNF219',
 'VWCE',
 'PHPT1',
 'LMNA',
 'SLC9A1',
 'C17orf89',
 'HRAS',
 'PPFIBP1',
 'UNC5B',
 'GADD45A',
 'PHLDA3',
 'TGFA',
 'ZNF337',
 'DDIT4',
 'PIDD1',
 'MLF2',
 'STAT3',
 'CAPN2',
 'HSD17B3',
 'PPM1J',
 'UQCC1',
 'PLK3

In [ ]:
expression.columns = expression.columns.str.replace(r"\s*\(\d+\)", "", regex=True)

X = expression[expression.columns.intersection(target_genes)]
y = expression['Mutation'].astype('category').cat.codes

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
models = {}

# Logistic Regression
from sklearn.linear_model import LogisticRegression
models['Logistic Regression'] = LogisticRegression()

# Support Vector Machines
from sklearn.svm import LinearSVC
models['Support Vector Machines'] = LinearSVC()

# Decision Trees
from sklearn.tree import DecisionTreeClassifier
models['Decision Trees'] = DecisionTreeClassifier()

# Random Forest
from sklearn.ensemble import RandomForestClassifier
models['Random Forest'] = RandomForestClassifier()

# Naive Bayes
from sklearn.naive_bayes import GaussianNB
models['Naive Bayes'] = GaussianNB()

# K-Nearest Neighbors
from sklearn.neighbors import KNeighborsClassifier
models['K-Nearest Neighbor'] = KNeighborsClassifier()
from sklearn.metrics import accuracy_score, precision_score, recall_score

accuracy, precision, recall = {}, {}, {}

for key in models.keys():
    
    # Fit the classifier
    models[key].fit(X_train, y_train)
    
    # Make predictions
    predictions = models[key].predict(X_test)
    
    # Calculate metrics
    accuracy[key] = accuracy_score(predictions, y_test)
    precision[key] = precision_score(predictions, y_test)
    recall[key] = recall_score(predictions, y_test)
import pandas as pd

df_model = pd.DataFrame(index=models.keys(), columns=['Accuracy', 'Precision', 'Recall'])
df_model['Accuracy'] = accuracy.values()
df_model['Precision'] = precision.values()
df_model['Recall'] = recall.values()

df_model

/Users/beatricecitterio/opt/anaconda3/envs/ml/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,Accuracy,Precision,Recall
Logistic Regression,0.832836,0.817259,0.889503
Support Vector Machines,0.800000,0.771574,0.873563
Decision Trees,0.794030,0.796954,0.844086
Random Forest,0.877612,0.913706,0.882353
Naive Bayes,0.743284,0.690355,0.844720
K-Nearest Neighbor,0.797015,0.807107,0.841270


In [ ]:
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2],
    'max_features': ['sqrt', 'log2']
}

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
import itertools

# Split the data once
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Generate all parameter combinations
param_combinations = list(itertools.product(
    param_grid['n_estimators'],
    param_grid['max_depth'],
    param_grid['min_samples_split'],
    param_grid['min_samples_leaf'],
    param_grid['max_features']
))

best_auc = 0
best_model = None
best_params = None

# Try each combination
for n_estimators, max_depth, min_samples_split, min_samples_leaf, max_features in param_combinations:
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        max_features=max_features,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    )
    model.fit(X_train, y_train)
    y_proba = model.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, y_proba)

    if auc > best_auc:
        best_auc = auc
        best_model = model
        best_params = {
            'n_estimators': n_estimators,
            'max_depth': max_depth,
            'min_samples_split': min_samples_split,
            'min_samples_leaf': min_samples_leaf,
            'max_features': max_features
        }

print("Best AUC:", best_auc)
print("Best Parameters:", best_params)



Best AUC: 0.9395460898992128
Best Parameters: {'n_estimators': 200, 'max_depth': 20, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'sqrt'}


In [ ]:
from sklearn.metrics import classification_report, roc_auc_score

# Predict on test set
y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:, 1]

# Performance metrics
print(classification_report(y_test, y_pred))
print("ROC AUC:", roc_auc_score(y_test, y_proba))


              precision    recall  f1-score   support

           0       0.90      0.82      0.86       138
           1       0.88      0.93      0.91       197

    accuracy                           0.89       335
   macro avg       0.89      0.88      0.88       335
weighted avg       0.89      0.89      0.89       335

ROC AUC: 0.9395460898992128
